# Benchmark Runner — Full 25-Claim Evaluation

Runs all 25 benchmark claims through the two-vote evaluator and reports accuracy.

| Tier | Claims | Expected verdict |
|------|--------|------------------|
| WELL_SUPPORTED (WS) | 8 | SUPPORTED |
| CONTESTED (CT) | 7 | CONTESTED |
| OVERCLAIMED (OC) | 10 | UNSUPPORTED / INSUFFICIENT_EVIDENCE |

**Flow:**
1. Seed ChromaDB with papers covering all 25 claim domains (skip on re-runs)
2. `run()` calls `evaluate_claim()` on each claim, writes timestamped JSON + CSV to `benchmarks/results/`
3. Summary table, per-tier accuracy, and failed claims are displayed below

In [ ]:
import sys, pathlib
# Kernel cwd is test/; parent is the project root
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print('Project root:', _root)

In [ ]:
import logging, pandas as pd
from dotenv import load_dotenv
load_dotenv()

from benchmarks.benchmark_runner import run, seed_chromadb

logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s — %(message)s')

## Step 1 — Seed ChromaDB (skip on re-runs)

Runs 13 MeSH-anchored PubMed queries covering all 25 claim domains and upserts results
into ChromaDB. Idempotent — safe to re-run but slow (~5 min). Skip if already populated.

In [ ]:
# Skip this cell if ChromaDB is already populated from a previous run.
seed_chromadb(disease='ipf', max_results=50)

## Step 2 — Run all 25 claims

Calls `evaluate_claim()` on each claim sequentially. Each call makes one Anthropic API
request. Exceptions are caught per claim so the run always completes all 25.
Results are written to `benchmarks/results/results_{timestamp}.json` and `.csv`.

In [ ]:
# seed=False — ChromaDB already seeded in cell above
rows = run(disease='ipf')
print(f'\nReturned {len(rows)} rows.')

## Step 3 — Full results table

In [ ]:
df = pd.DataFrame(rows)
df['correct_bool'] = df['correct'].astype(bool)
df['correct'] = df['correct_bool'].map({True: '✓', False: '✗'})

display_cols = [
    'claim_id', 'tier', 'expected_verdict', 'actual_verdict',
    'verdict', 'verdict_confidence', 'correct',
    'prior_support_score', 'contested_flags',
]
pd.set_option('display.max_colwidth', 60)
display(df[display_cols].style.set_properties(**{'text-align': 'left'}).hide(axis='index'))

## Step 4 — Accuracy by tier

In [ ]:
total   = len(df)
correct = df['correct_bool'].sum()
print(f'Overall: {correct}/{total} ({100*correct/total:.1f}%)\n')

for tier, g in df.groupby('tier'):
    n = len(g)
    c = g['correct_bool'].sum()
    print(f'  {tier:<14}: {c}/{n} ({100*c/n:.1f}%)')

## Step 5 — Failed claims

In [ ]:
failed = df[~df['correct_bool']]
if failed.empty:
    print('All claims passed.')
else:
    print(f'{len(failed)} failed claim(s):\n')
    fail_cols = [
        'claim_id', 'tier', 'expected_verdict', 'actual_verdict',
        'verdict', 'verdict_confidence', 'claim',
    ]
    display(failed[fail_cols].style.set_properties(**{'text-align': 'left'}).hide(axis='index'))